# Modélisation – H3c : via le bassin d'emploi
**Contributeurs :** Gabrielle (Ectoplasme) · **Date :** 04/11/2026, mis à jour le 05/11/2026

Notebook construit à partir du template commun de l'équipe. Il passe par `commun.py` pour tout ce qui doit être identique d'une hypothèse à l'autre : la table, le masque des lignes, les fenêtres, la naïve, `evaluer` et `tester`.

**Question :** l'évolution de l'emploi en N-1 améliore-t-elle la prévision du prix N ? Ce serait un premier signe qu'une variable « quand » (qui change par territoire et par année) fait mieux que les photos fixes.

**Variables :** effectifs salariés du secteur privé (Urssaf, au 31/12, par commune, regroupés au sous-bassin par le feature engineering, section « Emploi salarié privé ») :
- `evol_emploi_n2_n1` : évolution de l'effectif sur un an, en %, à champ constant (seules les cellules commune × activité connues les deux années comptent) ;
- `emploi_par_logement` : effectif rapporté au parc de logements, pour comparer des territoires de taille différente.

L'emploi ne dépend pas du type de bien : appartements et maisons d'un même sous-bassin ont la même valeur.

**Conclusion à transmettre à Chauve-Souris avant le mardi 10 novembre.**

## 0. Paramètres de l'hypothèse
- `HYPOTHESE` : préfixe du nom des modèles dans les tableaux.
- Deux blocs, dans l'ordre où ils entrent dans les modèles : d'abord l'**évolution** de l'emploi (la variable « quand », cœur de la question), puis le **niveau** d'emploi.
- `LOG` et `LOG1P` : liste de l'audit complémentaire (version 2). `emploi_par_logement` n'existait pas lors de l'audit : la même règle (asymétrie > 1) lui est appliquée en section 1, sur les années d'apprentissage de la première fenêtre seulement. `evol_emploi_n2_n1` ne passe jamais en log (valeurs négatives).
- `DECALAGE` : l'effectif retenu pour les prix de l'année N est celui du 31/12 de N-1 (convention « variable connue en N-1 »). Comme il n'est publié qu'au printemps de N, la variante prudente (31/12 de N-2) est testée en section 6.4.

In [1]:
HYPOTHESE = "H3c"

# Blocs de variables
BLOC_EVOLUTION = ["evol_emploi_n2_n1"]          # la variable « quand »
BLOC_NIVEAU = ["emploi_par_logement"]           # le niveau, comparable entre territoires
VARIABLES_BLOC = BLOC_EVOLUTION + BLOC_NIVEAU   # tous les blocs réunis (constat rapide, section 2)

# Tables du feature engineering (section Emploi), une par variante de calendrier :
# décalage de 1 an (version principale) et de 2 ans (variante prudente, section 6.4 c)
DECALAGE = 1
FICHIER_EMPLOI = "fe_emploi_decalage{}_sbv.parquet"

# Variables asymétriques de la table (audit complémentaire, version 2)
LOG = ["nb_ventes_n1", "densite_population", "niveau_vie_median",
       "taux3_vols_sans_violence", "taux3_vols_vehicules",
       "taux3_violences_sexuelles", "taux3_cambriolages", "taux3_violences_intrafamiliales",
       "taux_equip_commerces", "taux_equip_services", "taux_equip_sante", "taux_equip_sports",
       "part_sans_voiture", "part_moins40m2", "part_rsec", "part_apparts",
       "part_familles_monoparentales_pct_fam"]
LOG1P = ["taux3_vols_violents", "taux3_stupefiants", "taux_equip_tourisme", "taux_equip_transports"]

# Variables de l'hypothèse seulement ; prix_n1 passe en log avec la cible
VARIABLES_LOG = [v for v in LOG if v in VARIABLES_BLOC] + ["prix_n1"]
VARIABLES_LOG1P = [v for v in LOG1P if v in VARIABLES_BLOC]
print("log :", VARIABLES_LOG, "| log(1 + x) :", VARIABLES_LOG1P)

log : ['prix_n1'] | log(1 + x) : []


## 1. Import et chargement
`charger_table()` retire une seule fois les lignes à case vide (masque commun) : tous les modèles et la naïve sont jugés sur les mêmes lignes. `fenetres()` donne une fenêtre par année jugée (2023 et 2024), avec l'apprentissage sur les années d'avant.

**Emploi.** Le feature engineering (section « Emploi salarié privé ») produit l'emploi par sous-bassin et par année de prix, sans le brancher à la fusion : la table de modélisation ne le contient pas. La cellule le joint donc au sous-bassin et à l'année depuis `fe_emploi_decalage1_sbv.parquet`, sans toucher à l'index (`fe_emploi_decalage2_sbv.parquet` pour la variante de la section 6.4 c). Si l'emploi entre un jour dans la table, la fonction ne fait plus rien. Les lignes sans valeur d'emploi sont retirées, et leur nombre est affiché.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from copy import deepcopy
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import commun
from commun import CIBLE, charger_table, fenetres, plis_temporels, evaluer, tester
from gabarit_hierarchique import ModeleHierarchique
from gabarit_boosting import prevoir_boosting, GRILLE_REFERENCE   # gabarit commun (P3) 

def ajouter_emploi(table, decalage=DECALAGE):
    """Jointure du bloc emploi depuis la table du feature engineering (une par décalage), index conservé.
    Sans effet si l'emploi est déjà dans la table de modélisation (décalage 1)."""
    if all(v in table.columns for v in VARIABLES_BLOC) and decalage == DECALAGE:
        return table
    emploi = (pd.read_parquet(commun.DATA_OUT / FICHIER_EMPLOI.format(decalage))
              .set_index(["annee", "sous_bassin_de_vie"]))
    cle = pd.MultiIndex.from_arrays([table["annee"], table.index.get_level_values("sous_bassin_de_vie")])
    t = table.drop(columns=[v for v in VARIABLES_BLOC + ["part_emploi_secret"] if v in table.columns])
    for v in VARIABLES_BLOC + ["part_emploi_secret"]:
        if v in emploi.columns:
            t[v] = emploi[v].reindex(cle).to_numpy()
    vides = t[VARIABLES_BLOC].isna().any(axis=1)
    print(f"Emploi (décalage {decalage}) : {vides.sum()} lignes sans valeur ({vides.mean():.2%}), retirées ;",
          "par année :", t.loc[vides, "annee"].value_counts().sort_index().to_dict())
    return t[~vides]


table = ajouter_emploi(charger_table())
FENETRES = fenetres(table)

absentes = sorted(set(VARIABLES_BLOC + VARIABLES_LOG + VARIABLES_LOG1P) - set(table.columns))
assert not absentes, f"Variables absentes de la table : {absentes}"
print("Table :", table.shape)
for a, (app, ev) in FENETRES.items():
    print(f"Fenêtre {a} : apprentissage {sorted(app['annee'].unique())} ({len(app)} lignes) | jugée {a} ({len(ev)} lignes)")

Emploi (décalage 1) : 0 lignes sans valeur (0.00%), retirées ; par année : {}
Table : (23489, 42)
Fenêtre 2023 : apprentissage [np.int64(2020), np.int64(2021), np.int64(2022)] (7987 lignes) | jugée 2023 (2641 lignes)
Fenêtre 2024 : apprentissage [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] (10628 lignes) | jugée 2024 (2599 lignes)


In [3]:
# Règle de l'audit appliquée à emploi_par_logement : asymétrie > 1 -> log (ou log(1 + x) s'il y a des zéros).
# Mesurée sur l'apprentissage de la première fenêtre seulement, puis fixée pour tout le notebook.
app_regle = FENETRES[commun.ANNEES_JUGEES[0]][0]
for v in BLOC_NIVEAU:
    asym = app_regle[v].skew()
    if asym > 1:
        (VARIABLES_LOG1P if (app_regle[v] <= 0).any() else VARIABLES_LOG).append(v)
    print(f"{v} : asymétrie {asym:.2f}")
print("log :", VARIABLES_LOG, "| log(1 + x) :", VARIABLES_LOG1P)

emploi_par_logement : asymétrie 8.73
log : ['prix_n1', 'emploi_par_logement'] | log(1 + x) : []


## 2. Constat rapide
Distribution des variables du bloc et lien avec le prix (rho de Spearman), sur les années d'apprentissage de la première fenêtre seulement : l'année jugée n'est jamais regardée avant la modélisation.

In [4]:
app_constat = FENETRES[commun.ANNEES_JUGEES[0]][0]
print(app_constat[VARIABLES_BLOC].describe().round(2).to_string())

liens = []
for type_bien, e in app_constat.groupby("type_bien"):
    for v in VARIABLES_BLOC:
        liens.append({"type de bien": type_bien, "variable": v, "rho": stats.spearmanr(e[v], e[CIBLE]).statistic})
pd.DataFrame(liens).pivot(index="variable", columns="type de bien", values="rho").round(2)

       evol_emploi_n2_n1  emploi_par_logement
count            7987.00              7987.00
mean                2.32                 0.38
std                 6.20                 0.30
min               -60.44                 0.06
25%                -0.05                 0.21
50%                 1.99                 0.32
75%                 4.33                 0.45
max               153.38                 7.97


type de bien,appartement,maison
variable,,
emploi_par_logement,-0.05,0.17
evol_emploi_n2_n1,0.19,0.25


### 2.2 Redondances dans le bloc et lien avec l'écart au prix N-1
- **Redondances** : couples de variables du bloc dont |rho| dépasse 0,7 ; deux variables aussi liées portent la même information.
- **Écart au prix N-1** : `prix_n1` porte déjà le niveau du prix. Ce que le bloc peut apporter, c'est ce que `prix_n1` n'explique pas : l'évolution relative prix / prix N-1 − 1. On mesure son lien avec chaque variable (rho de Spearman).
- Années d'apprentissage de la première fenêtre seulement. Un même territoire revient chaque année : les rho décrivent, ils ne se testent pas.

In [5]:
couples = []
for type_bien, e in app_constat.groupby("type_bien"):
    rho = e[VARIABLES_BLOC].corr(method="spearman")
    for i, a in enumerate(VARIABLES_BLOC):
        for b in VARIABLES_BLOC[i + 1:]:
            if abs(rho.loc[a, b]) > 0.7:
                couples.append({"type de bien": type_bien, "variable 1": a, "variable 2": b, "rho": round(rho.loc[a, b], 2)})
print(pd.DataFrame(couples).to_string(index=False) if couples else "Aucun couple au-delà de 0,7")

evolution_app = app_constat[CIBLE] / app_constat["prix_n1"] - 1
liens_evolution = []
for type_bien, e in app_constat.groupby("type_bien"):
    for v in VARIABLES_BLOC:
        liens_evolution.append({"type de bien": type_bien, "variable": v,
                                "rho": stats.spearmanr(e[v], evolution_app.loc[e.index]).statistic})
pd.DataFrame(liens_evolution).pivot(index="variable", columns="type de bien", values="rho").round(2)

Aucun couple au-delà de 0,7


type de bien,appartement,maison
variable,,
emploi_par_logement,-0.04,-0.01
evol_emploi_n2_n1,0.08,0.11


## 3. Fonctions du notebook
Chaque modèle est lancé en une ligne : `lancer(nom, modèle, variables, log=..., grille=...)`.
- `nom` : le nom du modèle dans les tableaux (le préfixe de l'hypothèse est ajouté automatiquement) ;
- `modèle` : le modèle à utiliser (régression linéaire, Lasso, gradient boosting, modèle hiérarchique) ;
- `variables` : les colonnes de la table données au modèle ;
- `log=True` : la cible et les variables de `VARIABLES_LOG` passent en log, celles de `VARIABLES_LOG1P` en log(1 + x) ; à réserver aux modèles qui en ont besoin (régressions, Lasso, modèle hiérarchique), jamais aux arbres ;
- `grille` : les valeurs d'un réglage à tester (par exemple `alpha` pour le Lasso).

Ce que fait `lancer`, pour chacune des deux années jugées (2023, puis 2024) et pour chaque type de bien :
1. elle prend les années d'apprentissage, avant l'année jugée ;
2. elle met les variables à l'échelle et entraîne le modèle ; avec une grille, elle choisit d'abord le meilleur réglage, chaque année d'apprentissage servant tour à tour de validation (`plis_temporels` de `commun.py`) ;
3. elle prévoit le prix de l'année jugée, remis en €/m² si le modèle a appris en log ;
4. elle compare la prévision au prix réel et à la naïve (prix N = prix N-1) avec `evaluer` et `tester` de `commun.py`.

Elle affiche ensuite l'erreur du modèle (MAE, en €/m²), celle de la naïve, le gain sur la naïve (négatif : le modèle fait moins bien) et la part des sous-bassins où le modèle gagne. Elle garde les modèles entraînés (`ajustes`, pour la section 6) et leurs prévisions (`previsions`, pour la section 5.2). Relancer une cellule remplace les résultats du modèle au lieu de les ajouter une seconde fois.

Le modèle hiérarchique (`gabarit_hierarchique.py`) passe par une enveloppe : il n'est pas mis dans le pipeline (il met lui-même ses variables à l'échelle) et n'accepte pas de grille.

In [6]:
resultats, tests, ajustes, previsions = [], [], {}, {}


def en_log(X):
    """Variables asymétriques en log ; log(1 + x) pour celles qui ont des zéros."""
    X = X.copy()
    cols = [c for c in VARIABLES_LOG if c in X.columns]
    cols1p = [c for c in VARIABLES_LOG1P if c in X.columns]
    assert (X[cols] > 0).all().all(), "valeur nulle ou négative dans une variable de VARIABLES_LOG : la passer dans LOG1P"
    X[cols] = np.log(X[cols])
    X[cols1p] = np.log1p(X[cols1p])
    return X


class Hierarchique(RegressorMixin, BaseEstimator):
    """Enveloppe scikit-learn du modèle hiérarchique : log de la cible si demandé, prévisions en €/m²."""
    def __init__(self, modele=None, log=False):
        self.modele = modele
        self.log = log

    def fit(self, X, y):
        self.modele_ = deepcopy(self.modele).fit(X, np.log(y) if self.log else y)
        return self

    def predict(self, X):
        p = np.asarray(self.modele_.predict(X))
        return np.exp(p) if self.log else p


def lancer(nom, modele, variables, log=False, grille=None):
    """Ajuste un modèle par fenêtre et par type de bien, puis evaluer et tester ; affiche le résultat."""
    nom = f"{HYPOTHESE} – {nom}"
    # Relancer une cellule remplace les résultats du modèle au lieu de les ajouter une seconde fois
    resultats[:] = [r for r in resultats if r["modèle"] != nom]
    tests[:] = [t for t in tests if t["modèle"] != nom]
    for annee, (app, ev) in FENETRES.items():
        prevision = pd.Series(np.nan, index=ev.index)
        for type_bien, e in app.groupby("type_bien"):
            idx = ev.index[ev.index.get_level_values("type_bien") == type_bien]
            X_app, X_ev = e[variables], ev.loc[idx, variables]
            if isinstance(modele, ModeleHierarchique):
                assert not grille, "pas de grille pour le modèle hiérarchique"
                if log:
                    X_app, X_ev = en_log(X_app), en_log(X_ev)
                m = Hierarchique(modele, log).fit(X_app, e[CIBLE])
                prevision.loc[idx] = m.predict(X_ev)
                ajustes[(nom, annee, type_bien)] = (m, X_ev, ev.loc[idx, CIBLE])
                continue
            m = Pipeline([("echelle", StandardScaler()), ("modele", clone(modele))])
            prefixe = "modele__"
            if log:
                X_app, X_ev = en_log(X_app), en_log(X_ev)
                m = TransformedTargetRegressor(regressor=m, func=np.log, inverse_func=np.exp)
                prefixe = "regressor__modele__"
            if grille:
                # plis recalculés sur e (positions dans e, même ordre que X_app)
                m = GridSearchCV(m, {prefixe + k: v for k, v in grille.items()},
                                 cv=plis_temporels(e), scoring="neg_mean_absolute_error")
            m.fit(X_app, e[CIBLE])
            prevision.loc[idx] = m.predict(X_ev)
            ajustes[(nom, annee, type_bien)] = (m, X_ev, ev.loc[idx, CIBLE])
        assert prevision.notna().all(), f"{nom} : prévisions manquantes"
        previsions[(nom, annee)] = prevision          # gardée pour comparer les modèles entre eux (section 5.2)
        evaluer(nom, ev, prevision, resultats)
        tester(nom, ev, prevision, tests)

    # Résultat du modèle, affiché tout de suite : erreur du modèle, erreur de la naïve, gain
    res = pd.DataFrame(resultats)
    res = res[res["modèle"] == nom][["type de bien", "année jugée", "MAE (€/m²)", "MAE naïve (€/m²)",
                                     "gain sur la naïve (%)", "bassins où le modèle gagne (%)"]]
    print(nom, f"({len(variables)} variables)")
    print(res.round(1).to_string(index=False))
    return nom

## 4. Modèles
Du plus simple au plus complexe ; chaque étape ajoute un bloc à la précédente : prix N-1 seul, puis + évolution de l'emploi, puis + niveau d'emploi. Tous les modèles sont gardés, y compris ceux qui perdent. Les régressions et le Lasso sont faits en log, la référence aussi, pour une comparaison à égalité ; le gradient boosting n'a pas besoin du log.

Le gradient boosting a sa propre référence « prix N-1 seul » (4.4) : comparer un gradient boosting à une régression mélangerait l'apport du bloc et celui de la méthode (section 6.4).

In [7]:
# Blocs successifs : chaque étape ajoute un bloc à la précédente
REFERENCE_VARS = ["prix_n1"]
AVEC_EVOLUTION = REFERENCE_VARS + BLOC_EVOLUTION
BLOC_COMPLET = AVEC_EVOLUTION + BLOC_NIVEAU

### 4.1 Référence : prix N-1 seul
La naïve (prix N = prix N-1) est déjà dans chaque ligne du tableau (« MAE naïve »). Le modèle « prix N-1 seul » est la référence que le bloc doit battre (section 5.2).

In [8]:
REFERENCE = lancer("Régression linéaire – prix N-1 seul", LinearRegression(), REFERENCE_VARS, log=True)

H3c – Régression linéaire – prix N-1 seul (1 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       185.5             166.3                  -11.5                            45.6
      maison         2023       170.0             111.7                  -52.2                            31.0
 appartement         2024       221.5             148.0                  -49.7                            27.0
      maison         2024       214.9             129.2                  -66.4                            18.2


### 4.2 Régression linéaire : prix N-1 + emploi

In [9]:
EVOLUTION = lancer("Régression linéaire – prix N-1 + évolution emploi", LinearRegression(), AVEC_EVOLUTION, log=True)
COMPLET = lancer("Régression linéaire – prix N-1 + emploi (évolution + niveau)", LinearRegression(),
                 BLOC_COMPLET, log=True)

H3c – Régression linéaire – prix N-1 + évolution emploi (2 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       183.7             166.3                  -10.4                            46.2
      maison         2023       168.1             111.7                  -50.5                            31.2
 appartement         2024       214.4             148.0                  -44.9                            27.7
      maison         2024       209.8             129.2                  -62.4                            18.9
H3c – Régression linéaire – prix N-1 + emploi (évolution + niveau) (3 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       181.2             166.3                   -8.9                            46.5
      maison         2023       167.8             111.7                 

### 4.3 Lasso : quelles variables survivent
Le Lasso ramène à zéro les coefficients des variables qui n'apportent rien. `alpha` est réglé sur les plis temporels. Les valeurs testées sont petites : la cible est en log et les variables sont mises à l'échelle, les coefficients sont donc petits eux aussi.

Le Lasso a lui aussi sa propre référence « prix N-1 seul » : en réduisant les coefficients, il atténue la hausse apprise sur les années d'apprentissage, et ce gain-là ne doit pas être attribué à l'emploi (section 6.4 b).

In [10]:
GRILLE_LASSO = {"alpha": [0.0001, 0.001, 0.01, 0.1]}
LASSO_REF = lancer("Lasso – prix N-1 seul", Lasso(max_iter=10_000), REFERENCE_VARS, log=True, grille=GRILLE_LASSO)
LASSO = lancer("Lasso – prix N-1 + emploi (évolution + niveau)", Lasso(max_iter=10_000), BLOC_COMPLET, log=True,
               grille=GRILLE_LASSO)

H3c – Lasso – prix N-1 seul (1 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       174.2             166.3                   -4.7                            47.7
      maison         2023       169.8             111.7                  -52.0                            31.0
 appartement         2024       201.6             148.0                  -36.2                            28.3
      maison         2024       213.2             129.2                  -65.0                            18.1
H3c – Lasso – prix N-1 + emploi (évolution + niveau) (3 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       174.2             166.3                   -4.7                            47.7
      maison         2023       167.1             111.7                  -49.6                            31.3
 ap

### 4.4 Gradient boosting (gabarit commun P3)
Le gabarit de l'équipe (`gabarit_boosting.py`) remplace le modèle provisoire : même modèle, même grille et même réglage pour toutes les hypothèses, donc une différence entre hypothèses vient des variables et non des réglages.

- Il apprend l'**évolution** du prix, log(prix / prix N-1), puis rend une prévision en €/m² (prix N-1 × évolution prévue) : un arbre ne prévoit que des moyennes de prix déjà vus, il garde ainsi la précision du prix N-1 de chaque territoire.
- Réglage sur les plis temporels de `commun.py` (grille de référence, 8 combinaisons), puis réapprentissage sur toute la fenêtre ; pas d'arrêt précoce (il mélangerait les années).
- Pas de log ni de mise à l'échelle : la fonction `lancer` du template n'est donc pas utilisée. `lancer_boosting` fait la même chose avec le gabarit : boucle des fenêtres, `evaluer`, `tester`, et garde les modèles (`ajustes`) et les prévisions (`previsions`) pour les sections 5.2 et 6.

La référence « prix N-1 seul » est faite avec le même gabarit (section 6.4 b).

In [11]:
def lancer_boosting(nom, variables):
    """Comme lancer, avec le gabarit P3 : un modèle par fenêtre et par type de bien, puis evaluer et tester."""
    nom = f"{HYPOTHESE} – {nom}"
    # Relancer la cellule remplace les résultats du modèle au lieu de les ajouter une seconde fois
    resultats[:] = [r for r in resultats if r["modèle"] != nom]
    tests[:] = [t for t in tests if t["modèle"] != nom]
    for annee, (app, ev) in FENETRES.items():
        prevision, modeles = prevoir_boosting(app, ev, variables)      # grille de référence, plis temporels
        for type_bien, m in modeles.items():
            idx = ev.index[ev.index.get_level_values("type_bien") == type_bien]
            ajustes[(nom, annee, type_bien)] = (m, ev.loc[idx, variables], ev.loc[idx, CIBLE])
            print(annee, type_bien, "réglage retenu :", m.best_params_)
        previsions[(nom, annee)] = prevision
        evaluer(nom, ev, prevision, resultats)
        tester(nom, ev, prevision, tests)

    res = pd.DataFrame(resultats)
    res = res[res["modèle"] == nom][["type de bien", "année jugée", "MAE (€/m²)", "MAE naïve (€/m²)",
                                     "gain sur la naïve (%)", "bassins où le modèle gagne (%)"]]
    print(nom, f"({len(variables)} variables)")
    print(res.round(1).to_string(index=False))
    return nom


BOOSTING_REF = lancer_boosting("Gradient boosting – prix N-1 seul", REFERENCE_VARS)
BOOSTING = lancer_boosting("Gradient boosting – prix N-1 + emploi (évolution + niveau)", BLOC_COMPLET)

2023 appartement réglage retenu : {'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 50}
2023 maison réglage retenu : {'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 50}
2024 appartement réglage retenu : {'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 50}
2024 maison réglage retenu : {'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 50}
H3c – Gradient boosting – prix N-1 seul (1 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       182.6             166.3                   -9.8                            45.8
      maison         2023       171.5             111.7                  -53.5                            31.2
 appartement         2024       222.8             148.0                  -50.5                            26.4
      maison         2024       216.7             129.2                  -67.7                            18.5
2023 appa

### 4.5 Modèle hiérarchique
Le même sous-bassin revient chaque année : le modèle hiérarchique (`gabarit_hierarchique.py`, P4) donne à chaque sous-bassin sa propre ordonnée à l'origine (un « effet territoire »), au lieu de traiter les lignes comme indépendantes. Un sous-bassin absent de l'apprentissage reçoit les seuls effets fixes. Il est fait en log, comme la régression, et il met lui-même ses variables à l'échelle.

Deux modèles : la référence « prix N-1 seul » et le bloc complet, comparés entre eux en section 6.4 b (même méthode, seul l'emploi change).

**À lire dans le diagnostic** sous les résultats :
- **convergé** : si l'optimiseur n'a pas convergé, les coefficients sont moins sûrs (avertissements de statsmodels comptés à côté) ;
- **ICC**, la part de la variance due à l'effet territoire : avec `prix_n1`, elle est presque nulle (voir `test.py`), car le prix N-1 porte déjà le niveau du territoire. Un ICC proche de 0 veut dire que le modèle revient à la régression linéaire ;
- **coefficients des effets fixes** (variables centrées-réduites, cible en log) : signe et stabilité de l'évolution de l'emploi d'une année à l'autre.

In [12]:
HIER_REF = lancer("Hiérarchique – prix N-1 seul", ModeleHierarchique(), REFERENCE_VARS, log=True)
HIERARCHIQUE = lancer("Hiérarchique – prix N-1 + emploi (évolution + niveau)", ModeleHierarchique(),
                      BLOC_COMPLET, log=True)

# Diagnostic de chaque ajustement (un par fenêtre et par type de bien)
diagnostic = []
for nom_h in (HIER_REF, HIERARCHIQUE):
    for annee in commun.ANNEES_JUGEES:
        for type_bien in ("appartement", "maison"):
            m, X_ev, _ = ajustes[(nom_h, annee, type_bien)]
            mh = m.modele_                                  # ModeleHierarchique ajusté (enveloppe Hierarchique)
            coefs = np.asarray(mh.resultat_.fe_params)
            diagnostic.append({"modèle": nom_h, "année jugée": annee, "type de bien": type_bien,
                               "convergé": mh.converge_, "avertissements": len(mh.avertissements_),
                               "ICC": mh.icc_,
                               "territoires jugés sans effet propre (%)": mh.part_groupes_inconnus(X_ev) * 100,
                               **{"coef. " + v: c for v, c in zip(["constante"] + mh.variables_, coefs)}})
pd.DataFrame(diagnostic).set_index(["modèle", "année jugée", "type de bien"]).round(4)

H3c – Hiérarchique – prix N-1 seul (1 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       185.4             166.3                  -11.5                            45.5
      maison         2023       170.0             111.7                  -52.2                            31.0
 appartement         2024       221.5             148.0                  -49.7                            27.0
      maison         2024       214.9             129.2                  -66.4                            18.2
H3c – Hiérarchique – prix N-1 + emploi (évolution + niveau) (3 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       181.2             166.3                   -8.9                            46.5
      maison         2023       167.9             111.7                  -50.3                      

convergé  \
modèle                                             année jugée type de bien             
H3c – Hiérarchique – prix N-1 seul                 2023        appartement      False   
                                                               maison           False   
                                                   2024        appartement       True   
                                                               maison           False   
H3c – Hiérarchique – prix N-1 + emploi (évoluti... 2023        appartement       True   
                                                               maison           False   
                                                   2024        appartement       True   
                                                               maison           False   

                                                                             avertissements  \
modèle                                             année jugée type de bien                   
H3c – Hiérarchique – prix N-1 seul                 2023        appartement                8   
                                                               maison                     9   
                                                   2024        appartement                3   
                                                               maison                     9   
H3c – Hiérarchique – prix N-1 + emploi (évoluti... 2023        appartement                5   
                                                               maison                     9   
                                                   2024        appartement                3   
                                                               maison                     9   

                                                                                ICC  \
modèle                                             année jugée type de bien           
H3c – Hiérarchique – prix N-1 seul                 2023        appartement   0.0063   
                                                               maison        0.0018   
                                                   2024        appartement   0.0000   
                                                               maison        0.0000   
H3c – Hiérarchique – prix N-1 + emploi (évoluti... 2023        appartement   0.0000   
                                                               maison        0.0015   
                                                   2024        appartement   0.0001   
                                                               maison        0.0000   

                                                                             territoires jugés sans effet propre (%)  \
modèle                                             année jugée type de bien                                            
H3c – Hiérarchique – prix N-1 seul                 2023        appartement                                    0.9292   
                                                               maison                                         0.0562   
                                                   2024        appartement                                    0.2430   
                                                               maison                                         0.0000   
H3c – Hiérarchique – prix N-1 + emploi (évoluti... 2023        appartement                                    0.9292   
                                                               maison                                         0.0562   
                                                   2024        appartement                                    0.2430   
                                                               maison                                         0.0000   

                                                                             coef. constante  \
modèle                                             année jugée type

## 5. Mesures et tests
Trois tableaux, du plus simple au plus exigeant :
1. **`res` : les erreurs.** MAE du modèle et de la naïve en €/m², gain sur la naïve en %, part des sous-bassins où le modèle gagne. C'est une description : un modèle peut avoir une MAE plus basse par hasard.
2. **`tst` : les tests contre la naïve.** Ils disent si l'écart avec la naïve est plus grand que le hasard.
   - **Écart de MAE avec la naïve** : erreur du modèle moins erreur de la naïve, en €/m² (négatif : le modèle fait mieux).
   - **IC 95 %** (bootstrap par bassin de vie) : la fourchette dans laquelle se trouve cet écart, en tenant compte du hasard. Si toute la fourchette est sous 0, le modèle fait vraiment mieux ; si elle contient 0, on ne peut pas conclure ; si elle est toute au-dessus de 0, il fait vraiment moins bien.
   - **p-value de Wilcoxon** : le modèle fait-il mieux (ou moins bien) dans la plupart des sous-bassins ? Avec plusieurs centaines de sous-bassins, elle est presque toujours très petite : on la lit avec la taille d'effet.
   - **Rang-bisériale** (taille d'effet) : de −1 (le modèle fait mieux partout) à +1 (il fait moins bien partout) ; autour de 0, pas de différence nette.
3. **`decision` : la règle de décision.** Une année est réussie si l'IC est entièrement sous 0, la rang-bisériale négative et le gain d'au moins 3 %. Un modèle est retenu s'il réussit la majorité des années jugées (ici, les deux).

In [13]:
res = pd.DataFrame(resultats)
res.round(2)

,modèle,type de bien,année jugée,MAE (€/m²),MAE en % du prix,MAE naïve (€/m²),gain sur la naïve (%),bassins où le modèle gagne (%)
0,H3c – Régression linéaire – prix N-1 seul,appartement,2023,185.46,7.07,166.33,-11.51,45.64
1,H3c – Régression linéaire – prix N-1 seul,maison,2023,170.00,7.76,111.71,-52.18,30.96
2,H3c – Régression linéaire – prix N-1 seul,appartement,2024,221.51,8.50,148.00,-49.67,26.97
3,H3c – Régression linéaire – prix N-1 seul,maison,2024,214.92,9.64,129.19,-66.36,18.24
4,H3c – Régression linéaire – prix N-1 + évoluti...,appartement,2023,183.70,7.00,166.33,-10.44,46.23
5,H3c – Régression linéaire – prix N-1 + évoluti...,maison,2023,168.13,7.69,111.71,-50.51,31.24
6,H3c – Régression linéaire – prix N-1 + évoluti...,appartement,2024,214.40,8.26,148.00,-44.86,27.70
7,H3c – Régression linéaire – prix N-1 + évoluti...,maison,2024,209.76,9.44,129.19,-62.37,18.86
8,H3c – Régression linéaire – prix N-1 + emploi ...,appartement,2023,181.16,6.94,166.33,-8.92,46.46
9,H3c – Régression linéaire – prix N-1 + emploi ...,maison,2023,167.85,7.68,111.71,-50.26,31.40


In [14]:
tst = pd.DataFrame(tests)
tst.round(3)

,modèle,type de bien,année jugée,écart de MAE avec la naïve (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale,bassins de vie
0,H3c – Régression linéaire – prix N-1 seul,appartement,2023,19.140,-2.677,45.770,0.004,0.114,726
1,H3c – Régression linéaire – prix N-1 seul,maison,2023,58.291,50.608,65.476,0.000,0.471,1606
2,H3c – Régression linéaire – prix N-1 seul,appartement,2024,73.509,59.615,87.685,0.000,0.606,692
3,H3c – Régression linéaire – prix N-1 seul,maison,2024,85.732,80.270,91.580,0.000,0.778,1605
4,H3c – Régression linéaire – prix N-1 + évoluti...,appartement,2023,17.370,-4.386,43.798,0.008,0.104,726
5,H3c – Régression linéaire – prix N-1 + évoluti...,maison,2023,56.421,48.568,63.543,0.000,0.465,1606
6,H3c – Régression linéaire – prix N-1 + évoluti...,appartement,2024,66.399,54.083,78.838,0.000,0.588,692
7,H3c – Régression linéaire – prix N-1 + évoluti...,maison,2024,80.570,75.475,85.997,0.000,0.772,1605
8,H3c – Régression linéaire – prix N-1 + emploi ...,appartement,2023,14.832,-5.625,39.136,0.013,0.098,726
9,H3c – Régression linéaire – prix N-1 + emploi ...,maison,2023,56.141,48.356,63.289,0.000,0.464,1606


### 5.1 Règle de décision contre la naïve

In [15]:
SEUIL_GAIN = 3      # % de gain sur la MAE de la comparaison, fixé avant les résultats


def regle_decision(res, tst, comparaison="naïve"):
    """Années réussies et décision, par modèle et type de bien."""
    cles = ["modèle", "type de bien", "année jugée"]
    regle = res[cles + [f"gain sur la {comparaison} (%)"]].merge(
        tst[cles + ["IC 95 % haut (€/m²)", "rang-bisériale"]], on=cles)
    regle["année réussie"] = ((regle["IC 95 % haut (€/m²)"] < 0) & (regle["rang-bisériale"] < 0)
                              & (regle[f"gain sur la {comparaison} (%)"] >= SEUIL_GAIN))
    decision = regle.groupby(["modèle", "type de bien"], sort=False)["année réussie"].agg(["sum", "count"])
    decision["retenu"] = decision["sum"] > decision["count"] / 2
    return decision.rename(columns={"sum": "années réussies", "count": "années jugées"})


regle_decision(res, tst)

années réussies  \
modèle                                             type de bien                    
H3c – Régression linéaire – prix N-1 seul          appartement                 0   
                                                   maison                      0   
H3c – Régression linéaire – prix N-1 + évolutio... appartement                 0   
                                                   maison                      0   
H3c – Régression linéaire – prix N-1 + emploi (... appartement                 0   
                                                   maison                      0   
H3c – Lasso – prix N-1 seul                        appartement                 0   
                                                   maison                      0   
H3c – Lasso – prix N-1 + emploi (évolution + ni... appartement                 0   
                                                   maison                      0   
H3c – Gradient boosting – prix N-1 seul            appartement                 0   
                                                   maison                      0   
H3c – Gradient boosting – prix N-1 + emploi (év... appartement                 0   
                                                   maison                      0   
H3c – Hiérarchique – prix N-1 seul                 appartement                 0   
                                                   maison                      0   
H3c – Hiérarchique – prix N-1 + emploi (évoluti... appartement                 0   
                                                   maison                      0   

                                                                 années jugées  \
modèle                                             type de bien                  
H3c – Régression linéaire – prix N-1 seul          appartement               2   
                                                   maison                    2   
H3c – Régression linéaire – prix N-1 + évolutio... appartement               2   
                                                   maison                    2   
H3c – Régression linéaire – prix N-1 + emploi (... appartement               2   
                                                   maison                    2   
H3c – Lasso – prix N-1 seul                        appartement               2   
                                                   maison                    2   
H3c – Lasso – prix N-1 + emploi (évolution + ni... appartement               2   
                                                   maison                    2   
H3c – Gradient boosting – prix N-1 seul            appartement               2   
                                                   maison                    2   
H3c – Gradient boosting – prix N-1 + emploi (év... appartement               2   
                                                   maison                    2   
H3c – Hiérarchique – prix N-1 seul                 appartement               2   
                                                   maison                    2   
H3c – Hiérarchique – prix N-1 + emploi (évoluti... appartement               2   
                                                   maison                    2   

                                                                 retenu  
modèle                                             type de bien          
H3c – Régression linéaire – prix N-1 seul          appartement    False  
                                                   maison         False  
H3c – Régression linéaire – prix N-1 + évolutio... appartement    False  
                                                   maison         False  
H3c – Régression linéaire – prix N-1 + emploi (... appartement    False  
                                                   maison         False  
H3c – Lasso – prix N-1 seul                        appartement    False  
                                                   maison         False  
H3c

**Comment lire le tableau des tests (exemple : régression « prix N-1 seul », maisons 2024).** L'écart de MAE avec la naïve est de +85,7 €/m² : le modèle se trompe en moyenne de 86 €/m² de plus que la naïve. L'intervalle du bootstrap va de +80,3 à +91,6 €/m², entièrement au-dessus de 0 : ce n'est pas le hasard, le modèle fait vraiment moins bien. La rang-bisériale de +0,78 dit que c'est le cas dans presque tous les sous-bassins (la naïve gagne dans 82 % d'entre eux).

**Résultats :** aucun modèle ne bat la naïve, ni en 2023 ni en 2024, pour aucun type de bien ; aucun n'est retenu.
- Les pertes vont de −4,7 % (Lasso, appartements 2023) à −67,7 % (gradient boosting, maisons 2024). Elles sont plus fortes en 2024 qu'en 2023 et pour les maisons que pour les appartements.
- Le seul intervalle qui contient 0 est celui des appartements en 2023, pour plusieurs modèles : cette année-là, on ne peut pas conclure qu'ils font moins bien que la naïve.
- Ce résultat n'est pas propre à l'emploi : « prix N-1 seul » perd déjà de 11,5 à 66,4 %. Les modèles ont appris pendant la hausse de 2020-2022 et la prolongent alors que le marché se retourne (section 6.1). Pour H3c, la question est donc de savoir si l'emploi **réduit** cet écart (section 5.2).

### 5.2 Le bloc apporte-t-il quelque chose au prix N-1 seul ?
La question propre à l'hypothèse : les variables du bloc améliorent-elles la prévision **par rapport au modèle « prix N-1 seul »** ?

On applique exactement les mêmes mesures et la même règle de décision, en remplaçant la naïve par la prévision du modèle « prix N-1 seul ». Les fonctions `evaluer` et `tester` lisent la prévision de comparaison dans la colonne `prix_n1` : on leur donne une copie de l'année jugée où cette colonne contient la prévision de la référence.

Lecture : un gain positif et un intervalle du bootstrap entièrement sous 0 veulent dire que le modèle fait vraiment mieux que la référence.

In [16]:
resultats_ref, tests_ref = [], []
modeles_compares = [n for n in dict.fromkeys(k[0] for k in previsions) if n != REFERENCE]

for nom in modeles_compares:
    for annee, (app, ev) in FENETRES.items():
        ev_ref = ev.copy()
        ev_ref["prix_n1"] = previsions[(REFERENCE, annee)]     # la comparaison se fait avec la référence
        evaluer(nom, ev_ref, previsions[(nom, annee)], resultats_ref)
        tester(nom, ev_ref, previsions[(nom, annee)], tests_ref)

# Mêmes tableaux que contre la naïve, avec « référence » à la place de « naïve »
res_ref = pd.DataFrame(resultats_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
tst_ref = pd.DataFrame(tests_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
display(res_ref[["modèle", "type de bien", "année jugée", "MAE (€/m²)", "MAE référence (€/m²)",
                 "gain sur la référence (%)", "bassins où le modèle gagne (%)"]].round(2))
display(tst_ref[["modèle", "type de bien", "année jugée", "écart de MAE avec la référence (€/m²)",
                 "IC 95 % bas (€/m²)", "IC 95 % haut (€/m²)", "p-value Wilcoxon", "rang-bisériale"]].round(3))

,modèle,type de bien,année jugée,MAE (€/m²),MAE référence (€/m²),gain sur la référence (%),bassins où le modèle gagne (%)
0,H3c – Régression linéaire – prix N-1 + évoluti...,appartement,2023,183.70,185.46,0.95,63.41
1,H3c – Régression linéaire – prix N-1 + évoluti...,maison,2023,168.13,170.00,1.10,67.87
2,H3c – Régression linéaire – prix N-1 + évoluti...,appartement,2024,214.40,221.51,3.21,77.64
3,H3c – Régression linéaire – prix N-1 + évoluti...,maison,2024,209.76,214.92,2.40,74.49
4,H3c – Régression linéaire – prix N-1 + emploi ...,appartement,2023,181.16,185.46,2.32,67.02
5,H3c – Régression linéaire – prix N-1 + emploi ...,maison,2023,167.85,170.00,1.27,68.15
6,H3c – Régression linéaire – prix N-1 + emploi ...,appartement,2024,211.18,221.51,4.66,69.02
7,H3c – Régression linéaire – prix N-1 + emploi ...,maison,2024,209.37,214.92,2.58,70.38
8,H3c – Lasso – prix N-1 seul,appartement,2023,174.15,185.46,6.10,51.10
9,H3c – Lasso – prix N-1 seul,maison,2023,169.84,170.00,0.09,62.36


,modèle,type de bien,année jugée,écart de MAE avec la référence (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale
0,H3c – Régression linéaire – prix N-1 + évoluti...,appartement,2023,-1.769,-2.439,-1.132,0.000,-0.298
1,H3c – Régression linéaire – prix N-1 + évoluti...,maison,2023,-1.870,-2.221,-1.549,0.000,-0.449
2,H3c – Régression linéaire – prix N-1 + évoluti...,appartement,2024,-7.110,-9.606,-5.199,0.000,-0.618
3,H3c – Régression linéaire – prix N-1 + évoluti...,maison,2024,-5.162,-5.816,-4.618,0.000,-0.605
4,H3c – Régression linéaire – prix N-1 + emploi ...,appartement,2023,-4.308,-6.809,-2.569,0.000,-0.432
5,H3c – Régression linéaire – prix N-1 + emploi ...,maison,2023,-2.151,-2.532,-1.802,0.000,-0.481
6,H3c – Régression linéaire – prix N-1 + emploi ...,appartement,2024,-10.333,-18.882,-4.680,0.000,-0.440
7,H3c – Régression linéaire – prix N-1 + emploi ...,maison,2024,-5.545,-6.356,-4.826,0.000,-0.531
8,H3c – Lasso – prix N-1 seul,appartement,2023,-11.313,-29.448,0.034,0.002,-0.125
9,H3c – Lasso – prix N-1 seul,maison,2023,-0.160,-0.197,-0.128,0.000,-0.450


#### Règle de décision contre la référence
Même règle que contre la naïve, avec le modèle « prix N-1 seul » à la place de la naïve.

In [17]:
regle_decision(res_ref, tst_ref, comparaison="référence")

années réussies  \
modèle                                             type de bien                    
H3c – Régression linéaire – prix N-1 + évolutio... appartement                 1   
                                                   maison                      0   
H3c – Régression linéaire – prix N-1 + emploi (... appartement                 1   
                                                   maison                      0   
H3c – Lasso – prix N-1 seul                        appartement                 1   
                                                   maison                      0   
H3c – Lasso – prix N-1 + emploi (évolution + ni... appartement                 1   
                                                   maison                      1   
H3c – Gradient boosting – prix N-1 seul            appartement                 0   
                                                   maison                      0   
H3c – Gradient boosting – prix N-1 + emploi (év... appartement                 2   
                                                   maison                      1   
H3c – Hiérarchique – prix N-1 seul                 appartement                 0   
                                                   maison                      0   
H3c – Hiérarchique – prix N-1 + emploi (évoluti... appartement                 1   
                                                   maison                      0   

                                                                 années jugées  \
modèle                                             type de bien                  
H3c – Régression linéaire – prix N-1 + évolutio... appartement               2   
                                                   maison                    2   
H3c – Régression linéaire – prix N-1 + emploi (... appartement               2   
                                                   maison                    2   
H3c – Lasso – prix N-1 seul                        appartement               2   
                                                   maison                    2   
H3c – Lasso – prix N-1 + emploi (évolution + ni... appartement               2   
                                                   maison                    2   
H3c – Gradient boosting – prix N-1 seul            appartement               2   
                                                   maison                    2   
H3c – Gradient boosting – prix N-1 + emploi (év... appartement               2   
                                                   maison                    2   
H3c – Hiérarchique – prix N-1 seul                 appartement               2   
                                                   maison                    2   
H3c – Hiérarchique – prix N-1 + emploi (évoluti... appartement               2   
                                                   maison                    2   

                                                                 retenu  
modèle                                             type de bien          
H3c – Régression linéaire – prix N-1 + évolutio... appartement    False  
                                                   maison         False  
H3c – Régression linéaire – prix N-1 + emploi (... appartement    False  
                                                   maison         False  
H3c – Lasso – prix N-1 seul                        appartement    False  
                                                   maison         False  
H3c – Lasso – prix N-1 + emploi (évolution + ni... appartement    False  
                                                   maison         False  
H3c – Gradient boosting – prix N-1 seul            appartement    False  
                                                   maison         False  
H3c – Gradient boosting – prix N-1 + emploi (év... appartement     True  
                                                   maison         False  
H3c – Hiérarchique – prix N-1 seul     

**Comment lire le tableau des tests (exemple : régression avec l'évolution de l'emploi, appartements 2024).** Ajouter l'évolution de l'emploi fait baisser l'erreur de 7,1 €/m² par rapport à « prix N-1 seul » (de 221,5 à 214,4 €/m², soit un gain de 3,2 %). L'intervalle du bootstrap va de −9,6 à −5,2 €/m², entièrement sous 0, et la rang-bisériale vaut −0,62 : l'amélioration est réelle et touche la grande majorité des sous-bassins (78 %). Les trois conditions sont remplies : l'année est réussie.

**Résultats contre le modèle « prix N-1 seul » :**
- **L'emploi améliore toujours un peu la prévision.** Pour la régression, avec l'évolution seule ou avec l'évolution et le niveau, les huit intervalles sont entièrement sous 0 et les rang-bisériales vont de −0,30 à −0,62. Mais le gain est petit : de 1,0 à 3,2 % avec l'évolution seule, de 1,3 à 4,7 % avec le bloc complet. Le seuil de 3 % n'est atteint que pour les appartements en 2024 : une année réussie sur deux, donc **non retenu** pour la régression.
- **L'évolution porte l'essentiel de l'apport.** Elle donne à elle seule la plus grande partie du gain ; le niveau d'emploi ajoute surtout pour les appartements (de 3,2 à 4,7 % en 2024).
- **Le Lasso et le gradient boosting ne se lisent pas ici.** Contre la régression, ils mélangent l'apport de l'emploi et celui de leur méthode : le Lasso « prix N-1 seul » gagne déjà 6,1 et 9,0 % pour les appartements, sans aucune variable d'emploi. Leur comparaison juste, contre leur propre référence, est en section 6.4 b.
- **Le modèle hiérarchique donne les mêmes résultats que la régression.** L'ICC est presque nul (moins de 0,01) : avec le prix N-1, l'effet propre à chaque sous-bassin n'apporte rien. Les avertissements de convergence viennent de là (la variance de l'effet territoire est estimée à 0) et ne signalent pas une erreur. *(À vérifier sur les chiffres après relance.)*

## 6. Interprétation

### 6.1 Contexte : le marché s'est-il retourné ?
Les modèles apprennent sur les années de hausse 2020-2022, puis sont jugés sur 2023 et 2024. Pour lire leurs résultats, on mesure l'évolution du prix d'une année sur l'autre (prix / prix N-1 − 1), en moyenne et en médiane, par année et par type de bien, jusqu'à 2024. L'année 2025, réservée au test final, n'est pas regardée.

In [18]:
jusqua_2024 = table[table["annee"] <= commun.ANNEE_VALIDATION]
evolution = (jusqua_2024[CIBLE] / jusqua_2024["prix_n1"] - 1) * 100
retournement = (evolution.groupby([jusqua_2024["annee"], jusqua_2024.index.get_level_values("type_bien")])
                .agg(["mean", "median"]).round(1)
                .rename(columns={"mean": "évolution moyenne (%)", "median": "évolution médiane (%)"}))
retournement.unstack()

évolution moyenne (%)        évolution médiane (%)       
type_bien           appartement maison           appartement maison
annee                                                              
2017                        2.2    1.8                   2.1    2.0
2018                        1.4    2.1                   1.5    2.1
2019                        3.2    2.3                   3.0    2.3
2020                        5.2    4.7                   5.1    4.7
2021                        8.0    7.9                   7.5    7.7
2022                        8.8    9.6                   8.2    9.1
2023                        2.9    1.3                   2.9    0.8
2024                       -0.8   -2.4                  -1.5   -2.9

**Lecture.** Les années d'apprentissage sont des années de forte hausse : de +5 à +10 % par an entre 2020 et 2022. Puis la hausse ralentit nettement en 2023 (+2,9 % pour les appartements, +1,3 % pour les maisons en moyenne), et les prix baissent en 2024 (−0,8 % et −2,4 %). Les maisons se retournent plus tôt et plus fort que les appartements. C'est ce qui explique que tous les modèles surestiment les prix (erreur signée positive, section 6.3) et perdent contre la naïve, davantage en 2024 et pour les maisons.

### 6.2 Importance des variables
Importance par permutation, sur l'année jugée : on mélange au hasard les valeurs d'une variable entre les sous-bassins, 10 fois, et on mesure de combien l'erreur du modèle change (moyenne et écart-type). Une hausse : la variable était utile ; 0 : le modèle ne s'en sert pas ; une baisse : sur cette année, la variable dessert le modèle. Une variable compte si sa valeur dépasse deux écarts-types et au moins 1 €/m² (en dessous, l'effet est négligeable même s'il est stable).

Pour H3c, le tableau est donné pour **toutes** les combinaisons (2 années × 2 types de bien) du modèle complet de régression et du gradient boosting : la question porte sur le rôle de l'évolution de l'emploi, pas sur un seul cas.

In [19]:
lignes_imp = []
for nom_imp in (COMPLET, BOOSTING):
    for annee in commun.ANNEES_JUGEES:
        for type_bien in ("appartement", "maison"):
            m, X_ev, y_ev = ajustes[(nom_imp, annee, type_bien)]
            imp = permutation_importance(m, X_ev, y_ev, scoring="neg_mean_absolute_error",
                                         n_repeats=10, random_state=commun.GRAINE)
            for v, moy, et in zip(X_ev.columns, imp.importances_mean, imp.importances_std):
                lignes_imp.append({"modèle": nom_imp, "année jugée": annee, "type de bien": type_bien,
                                   "variable": v, "hausse de la MAE (€/m²)": moy, "écart-type (€/m²)": et,
                                   "compte (> 2 écarts-types et ≥ 1 €/m²)": (moy > 2 * et) and (moy >= 1)})
importance = pd.DataFrame(lignes_imp)
importance.round(1)

,modèle,année jugée,type de bien,variable,hausse de la MAE (€/m²),écart-type (€/m²),compte (> 2 écarts-types et ≥ 1 €/m²)
0,H3c – Régression linéaire – prix N-1 + emploi ...,2023,appartement,prix_n1,1589.0,29.8,True
1,H3c – Régression linéaire – prix N-1 + emploi ...,2023,appartement,evol_emploi_n2_n1,-0.7,0.3,False
2,H3c – Régression linéaire – prix N-1 + emploi ...,2023,appartement,emploi_par_logement,2.5,0.2,True
3,H3c – Régression linéaire – prix N-1 + emploi ...,2023,maison,prix_n1,1132.7,16.2,True
4,H3c – Régression linéaire – prix N-1 + emploi ...,2023,maison,evol_emploi_n2_n1,-0.3,0.1,False
5,H3c – Régression linéaire – prix N-1 + emploi ...,2023,maison,emploi_par_logement,0.4,0.1,False
6,H3c – Régression linéaire – prix N-1 + emploi ...,2024,appartement,prix_n1,1563.5,31.0,True
7,H3c – Régression linéaire – prix N-1 + emploi ...,2024,appartement,evol_emploi_n2_n1,-0.7,0.3,False
8,H3c – Régression linéaire – prix N-1 + emploi ...,2024,appartement,emploi_par_logement,3.7,0.5,True
9,H3c – Régression linéaire – prix N-1 + emploi ...,2024,maison,prix_n1,1035.2,18.3,True


**Lecture.** `prix_n1` domine de très loin : le mélanger fait monter l'erreur de 1 000 à 1 600 €/m². Pour l'emploi, l'importance est presque nulle.
- **Évolution de l'emploi** : son importance est nulle, voire légèrement négative (de −0,3 à −0,7 €/m² pour la régression, non significative pour le gradient boosting). Mélanger l'évolution entre les sous-bassins d'une même année ne dégrade pas la prévision : le modèle ne s'en sert pas pour distinguer les territoires.
- **Niveau d'emploi** : un apport faible mais stable pour les appartements dans la régression (+2,5 €/m² en 2023, +3,7 €/m² en 2024), à peine 1 €/m² pour les maisons, et rien de net dans le gradient boosting.

Pourtant, l'évolution améliore la prévision en 5.2. Les deux résultats se concilient : la permutation garde la moyenne de l'année et ne mesure que ce qui distingue les territoires entre eux. Le gain de l'évolution ne vient donc pas des écarts entre territoires, mais de son **niveau moyen de l'année**. L'emploi privé a ralenti au niveau national (+2,0 % au 31/12/2022, +0,5 % au 31/12/2023, contre +2,3 % en moyenne sur l'apprentissage) ; avec un coefficient positif, ce ralentissement fait baisser toutes les prévisions et corrige une partie de la surestimation (sections 6.4 a et d).

### 6.3 Analyse des erreurs selon le contrôle qualité
L'erreur est-elle plus grande là où les données sont estimées ou complétées ? Pour chaque variable de contrôle qualité : groupe « part nulle », puis quartiles parmi les parts non nulles. Par groupe : erreur moyenne en €/m² et en % du prix (l'erreur en €/m² grandit avec le niveau des prix), et erreur moyenne signée (positive : le modèle surestime). Rho de Spearman entre la variable et l'erreur, en €/m² et en %.

Pour H3c, une quatrième variable de contrôle s'ajoute : `part_emploi_secret`, la part des cellules commune × activité masquées par le secret statistique dans le sous-bassin. Le modèle étudié est choisi dans la cellule (par défaut : le modèle complet de régression, 2024, maisons).

In [20]:
choix = (COMPLET, 2024, "maison")      # modèle, année jugée, type de bien étudiés (à adapter)
CONTROLE = ["part_indicateurs_estimes", "part_logements_taux_impute", "part_menages_couverts"]
CONTROLE += [v for v in ["part_emploi_secret"] if v in table.columns]
m, X_ev, y_ev = ajustes[choix]
erreur = pd.Series(m.predict(X_ev), index=X_ev.index) - y_ev       # signée : positive = prix surestimé
erreur_pct = erreur.abs() / y_ev * 100                             # erreur absolue en % du prix réel
controle = table.loc[X_ev.index, CONTROLE]

for v in CONTROLE:
    x = controle[v]
    groupe = pd.Series("part nulle", index=x.index)
    non_nul = x > 0
    if non_nul.sum() >= 4:
        groupe[non_nul] = pd.qcut(x[non_nul], 4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop").astype(str)
    tableau = pd.DataFrame({"MAE (€/m²)": erreur.abs(), "erreur en % du prix": erreur_pct,
                            "erreur signée (€/m²)": erreur}).groupby(groupe).mean()
    tableau["lignes"] = groupe.value_counts()
    rho, p = stats.spearmanr(x, erreur.abs())
    rho_pct, _ = stats.spearmanr(x, erreur_pct)
    print(f"\n{v} : rho avec l'erreur en €/m² = {rho:.2f} (p = {p:.3f}) | rho avec l'erreur en % = {rho_pct:.2f}")
    print(tableau.round(1).to_string())


part_indicateurs_estimes : rho avec l'erreur en €/m² = -0.16 (p = 0.000) | rho avec l'erreur en % = -0.03
            MAE (€/m²)  erreur en % du prix  erreur signée (€/m²)  lignes
Q1               266.1                  9.4                 256.3     439
Q2               182.2                  9.2                 159.9     439
Q3               183.5                  9.3                 161.8     438
Q4               195.0                  9.6                 169.6     439
part nulle       432.4                 10.3                 432.4      21

part_logements_taux_impute : rho avec l'erreur en €/m² = -0.32 (p = 0.000) | rho avec l'erreur en % = -0.07
            MAE (€/m²)  erreur en % du prix  erreur signée (€/m²)  lignes
Q1               230.2                  9.4                 221.1     346
Q2               176.6                  8.9                 149.1     345
Q3               160.9                  9.1                 134.5     345
Q4               150.0                  9.8 

**Lecture (régression avec l'emploi, maisons 2024).** L'erreur signée est positive dans tous les groupes (de +88 à +432 €/m²) : le modèle surestime partout, conséquence du retournement (6.1).
- **Délinquance estimée et LOVAC complété** : l'erreur en €/m² est plus forte dans les sous-bassins sans aucune donnée estimée (432 et 314 €/m², contre 150 à 266 €/m² ailleurs). Mais en % du prix, l'écart disparaît (rho de −0,03 et −0,07). Ces sous-bassins sont surtout les plus grands et les plus chers, où les données sont complètes : l'erreur suit le niveau des prix, pas la qualité des données.
- **Ménages couverts par Filosofi** : aucun lien net (rho de 0,07 en €/m², 0,00 en %).
- **Emploi sous secret** : 1 713 lignes sur 1 776 n'ont aucune cellule masquée, et le lien avec l'erreur est nul (rho = 0,02, p = 0,35). Le secret statistique de l'Urssaf ne dégrade pas la prévision.

**Conclusion :** l'erreur n'est pas concentrée dans les territoires aux données estimées ou complétées. Ce n'est pas une limite à déclarer pour H3c.

### 6.4 Analyses propres à H3c
**a. Coefficient de l'évolution de l'emploi.** La feuille de route demande de regarder ce coefficient. Coefficient standardisé de la régression en log, prix N-1 + emploi : la hausse de log(prix) pour un écart-type de plus de la variable, toutes choses égales par ailleurs. Positif attendu : plus d'emplois en N-1, prix plus élevé en N. Le coefficient de `prix_n1` sert de repère d'échelle. Coefficients du Lasso à côté : une variable ramenée à 0 n'apporte rien.

In [21]:
lignes_coef = []
for nom_coef in (COMPLET, LASSO):
    for annee in commun.ANNEES_JUGEES:
        for type_bien in ("appartement", "maison"):
            m, X_ev, _ = ajustes[(nom_coef, annee, type_bien)]
            m = m.best_estimator_ if hasattr(m, "best_estimator_") else m
            coefs = m.regressor_.named_steps["modele"].coef_
            lignes_coef.append({"modèle": nom_coef, "année jugée": annee, "type de bien": type_bien,
                                **dict(zip(X_ev.columns, coefs))})
pd.DataFrame(lignes_coef).set_index(["modèle", "année jugée", "type de bien"]).round(4)

prix_n1  \
modèle                                             année jugée type de bien            
H3c – Régression linéaire – prix N-1 + emploi (... 2023        appartement    0.5292   
                                                               maison         0.4661   
                                                   2024        appartement    0.5304   
                                                               maison         0.4689   
H3c – Lasso – prix N-1 + emploi (évolution + ni... 2023        appartement    0.5199   
                                                               maison         0.4651   
                                                   2024        appartement    0.5210   
                                                               maison         0.4594   

                                                                             evol_emploi_n2_n1  \
modèle                                             année jugée type de bien                      
H3c – Régression linéaire – prix N-1 + emploi (... 2023        appartement              0.0074   
                                                               maison                   0.0033   
                                                   2024        appartement              0.0082   
                                                               maison                   0.0062   
H3c – Lasso – prix N-1 + emploi (évolution + ni... 2023        appartement              0.0000   
                                                               maison                   0.0025   
                                                   2024        appartement              0.0000   
                                                               maison                   0.0000   

                                                                             emploi_par_logement  
modèle                                             année jugée type de bien                       
H3c – Régression linéaire – prix N-1 + emploi (... 2023        appartement               -0.0022  
                                                               maison                    -0.0008  
                                                   2024        appartement               -0.0059  
                                                               maison                    -0.0029  
H3c – Lasso – prix N-1 + emploi (évolution + ni... 2023        appartement               -0.0000  
                                                               maison                    -0.0000  
                                                   2024        appartement               -0.0000  
                                                               maison                    -0.0000

**b. Chaque méthode contre son propre « prix N-1 seul ».** En 5.2, tous les modèles sont jugés contre la régression « prix N-1 seul ». Or le Lasso et le gradient boosting peuvent battre cette régression sans l'emploi, par leur seule méthode : le Lasso en réduisant les coefficients (il atténue la hausse apprise sur les années d'apprentissage), le gradient boosting en apprenant l'évolution plutôt que le niveau. La comparaison juste oppose chaque méthode à sa propre référence « prix N-1 seul », avec le même réglage : seul le bloc emploi change. Mêmes mesures, même règle.

In [22]:
resultats_meth, tests_meth = [], []
for candidat, reference in [(LASSO, LASSO_REF), (BOOSTING, BOOSTING_REF)]:
    for annee, (app, ev) in FENETRES.items():
        ev_ref = ev.copy()
        ev_ref["prix_n1"] = previsions[(reference, annee)]     # la comparaison se fait avec la référence de la méthode
        evaluer(candidat, ev_ref, previsions[(candidat, annee)], resultats_meth)
        tester(candidat, ev_ref, previsions[(candidat, annee)], tests_meth)
res_meth = pd.DataFrame(resultats_meth).rename(columns=lambda c: c.replace("naïve", "référence"))
tst_meth = pd.DataFrame(tests_meth).rename(columns=lambda c: c.replace("naïve", "référence"))
display(res_meth[["modèle", "type de bien", "année jugée", "MAE (€/m²)", "MAE référence (€/m²)",
                  "gain sur la référence (%)"]].round(2))
display(tst_meth[["modèle", "type de bien", "année jugée", "IC 95 % bas (€/m²)", "IC 95 % haut (€/m²)",
                  "rang-bisériale"]].round(3))
regle_decision(res_meth, tst_meth, comparaison="référence")

,modèle,type de bien,année jugée,MAE (€/m²),MAE référence (€/m²),gain sur la référence (%)
0,H3c – Lasso – prix N-1 + emploi (évolution + n...,appartement,2023,174.15,174.15,0.00
1,H3c – Lasso – prix N-1 + emploi (évolution + n...,maison,2023,167.07,169.84,1.63
2,H3c – Lasso – prix N-1 + emploi (évolution + n...,appartement,2024,201.55,201.55,0.00
3,H3c – Lasso – prix N-1 + emploi (évolution + n...,maison,2024,198.31,213.16,6.97
4,H3c – Gradient boosting – prix N-1 + emploi (é...,appartement,2023,177.67,182.58,2.69
5,H3c – Gradient boosting – prix N-1 + emploi (é...,maison,2023,165.07,171.45,3.73
6,H3c – Gradient boosting – prix N-1 + emploi (é...,appartement,2024,210.34,222.78,5.58
7,H3c – Gradient boosting – prix N-1 + emploi (é...,maison,2024,202.21,216.66,6.67


,modèle,type de bien,année jugée,IC 95 % bas (€/m²),IC 95 % haut (€/m²),rang-bisériale
0,H3c – Lasso – prix N-1 + emploi (évolution + n...,appartement,2023,0.000,0.000,NaN
1,H3c – Lasso – prix N-1 + emploi (évolution + n...,maison,2023,-3.254,-2.317,-0.519
2,H3c – Lasso – prix N-1 + emploi (évolution + n...,appartement,2024,0.000,0.000,NaN
3,H3c – Lasso – prix N-1 + emploi (évolution + n...,maison,2024,-17.625,-12.556,-0.447
4,H3c – Gradient boosting – prix N-1 + emploi (é...,appartement,2023,-7.737,-2.231,-0.182
5,H3c – Gradient boosting – prix N-1 + emploi (é...,maison,2023,-7.888,-5.088,-0.339
6,H3c – Gradient boosting – prix N-1 + emploi (é...,appartement,2024,-17.791,-7.219,-0.351
7,H3c – Gradient boosting – prix N-1 + emploi (é...,maison,2024,-16.280,-12.497,-0.535


années réussies  \
modèle                                             type de bien                    
H3c – Lasso – prix N-1 + emploi (évolution + ni... appartement                 0   
                                                   maison                      1   
H3c – Gradient boosting – prix N-1 + emploi (év... appartement                 1   
                                                   maison                      2   

                                                                 années jugées  \
modèle                                             type de bien                  
H3c – Lasso – prix N-1 + emploi (évolution + ni... appartement               2   
                                                   maison                    2   
H3c – Gradient boosting – prix N-1 + emploi (év... appartement               2   
                                                   maison                    2   

                                                                 retenu  
modèle                                             type de bien          
H3c – Lasso – prix N-1 + emploi (évolution + ni... appartement    False  
                                                   maison         False  
H3c – Gradient boosting – prix N-1 + emploi (év... appartement    False  
                                                   maison          True

**c. Variante prudente du calendrier (décalage de 2 ans).** L'effectif du 31/12 de N-1 n'est publié qu'au printemps de N. La variante prend l'effectif du 31/12 de N-2 (et son évolution depuis N-3) : une information réellement disponible au début de l'année N. Si l'apport du bloc disparaît avec ce décalage, il faut le déclarer comme limite. Fichier `fe_emploi_decalage2_sbv.parquet`, produit par le feature engineering. Les lignes peuvent différer un peu de la version principale (cases vides différentes) : la référence est donc relancée sur les mêmes lignes.

In [23]:
fichier_d2 = commun.DATA_OUT / FICHIER_EMPLOI.format(2)
if fichier_d2.exists():
    table_d2 = ajouter_emploi(charger_table(), decalage=2)
    FENETRES_PRINCIPALES, FENETRES = FENETRES, fenetres(table_d2)
    REF_D2 = lancer("Régression linéaire – prix N-1 seul (lignes décalage 2)", LinearRegression(),
                    REFERENCE_VARS, log=True)
    COMPLET_D2 = lancer("Régression linéaire – prix N-1 + emploi, décalage 2", LinearRegression(),
                        BLOC_COMPLET, log=True)
    res_d2, tst_d2 = [], []
    for annee, (app, ev) in FENETRES.items():
        ev_ref = ev.copy()
        ev_ref["prix_n1"] = previsions[(REF_D2, annee)]
        evaluer(COMPLET_D2, ev_ref, previsions[(COMPLET_D2, annee)], res_d2)
        tester(COMPLET_D2, ev_ref, previsions[(COMPLET_D2, annee)], tst_d2)
    FENETRES = FENETRES_PRINCIPALES                                   # retour à la version principale
    for annee in commun.ANNEES_JUGEES:          # lignes différentes : hors des comparaisons de la section 5.2
        previsions.pop((REF_D2, annee)); previsions.pop((COMPLET_D2, annee))
    res_d2 = pd.DataFrame(res_d2).rename(columns=lambda c: c.replace("naïve", "référence"))
    tst_d2 = pd.DataFrame(tst_d2).rename(columns=lambda c: c.replace("naïve", "référence"))
    display(res_d2[["modèle", "type de bien", "année jugée", "gain sur la référence (%)"]].round(2))
    display(regle_decision(res_d2, tst_d2, comparaison="référence"))
else:
    print(f"{fichier_d2.name} absent : relancer le feature engineering (section Emploi).")

Emploi (décalage 2) : 0 lignes sans valeur (0.00%), retirées ; par année : {}
H3c – Régression linéaire – prix N-1 seul (lignes décalage 2) (1 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       185.5             166.3                  -11.5                            45.6
      maison         2023       170.0             111.7                  -52.2                            31.0
 appartement         2024       221.5             148.0                  -49.7                            27.0
      maison         2024       214.9             129.2                  -66.4                            18.2
H3c – Régression linéaire – prix N-1 + emploi, décalage 2 (3 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       181.9             166.3                   -9.4                            46

,modèle,type de bien,année jugée,gain sur la référence (%)
0,"H3c – Régression linéaire – prix N-1 + emploi,...",appartement,2023,1.91
1,"H3c – Régression linéaire – prix N-1 + emploi,...",maison,2023,-0.83
2,"H3c – Régression linéaire – prix N-1 + emploi,...",appartement,2024,1.48
3,"H3c – Régression linéaire – prix N-1 + emploi,...",maison,2024,-1.30


années réussies  \
modèle                                             type de bien                    
H3c – Régression linéaire – prix N-1 + emploi, ... appartement                 0   
                                                   maison                      0   

                                                                 années jugées  \
modèle                                             type de bien                  
H3c – Régression linéaire – prix N-1 + emploi, ... appartement               2   
                                                   maison                    2   

                                                                 retenu  
modèle                                             type de bien          
H3c – Régression linéaire – prix N-1 + emploi, ... appartement    False  
                                                   maison         False

**d. Le gain dépend-il de l'évolution de l'emploi ?** Si l'emploi aide, le gain sur « prix N-1 seul » devrait se concentrer dans les territoires où l'emploi a le plus bougé (forte baisse ou forte hausse). Écart d'erreur absolue (modèle complet − référence) par quintile de `evol_emploi_n2_n1`, sur l'année jugée : négatif, le bloc aide dans ce groupe.

In [24]:
lignes_q = []
for annee, (app, ev) in FENETRES.items():
    gain = ((previsions[(COMPLET, annee)] - ev[CIBLE]).abs()
            - (previsions[(REFERENCE, annee)] - ev[CIBLE]).abs())
    quintile = pd.qcut(ev["evol_emploi_n2_n1"], 5, labels=["Q1 (baisse)", "Q2", "Q3", "Q4", "Q5 (hausse)"])
    for (tb, q), g in gain.groupby([ev.index.get_level_values("type_bien"), quintile], observed=True):
        lignes_q.append({"année jugée": annee, "type de bien": tb, "quintile évolution emploi": q,
                         "écart d'erreur moyen (€/m²)": g.mean(), "lignes": len(g)})
pd.DataFrame(lignes_q).pivot_table(index=["type de bien", "quintile évolution emploi"],
                                   columns="année jugée", values="écart d'erreur moyen (€/m²)",
                                   observed=True).round(1)

année jugée                             2023  2024
type de bien quintile évolution emploi            
appartement  Q1 (baisse)                -5.3 -13.8
             Q2                         -3.1 -10.2
             Q3                         -4.6  -9.5
             Q4                         -3.8  -9.4
             Q5 (hausse)                -5.0  -9.3
maison       Q1 (baisse)                -4.5  -9.8
             Q2                         -3.0  -8.8
             Q3                         -2.5  -7.0
             Q4                         -1.5  -5.0
             Q5 (hausse)                 1.2   2.5

**Comment lire 6.4 :**
- **a. Coefficients** : ils sont standardisés, en log. Un coefficient de 0,008 veut dire qu'un écart-type de plus d'évolution de l'emploi (environ 6 points) augmente le prix prévu d'environ 0,8 %. À comparer à `prix_n1` (environ 0,5).
- **b. Chaque méthode contre sa propre référence** : même lecture qu'en 5.2.
- **c. Calendrier prudent** : on compare le gain sur la référence avec celui de la version principale.
- **d. Quintiles** : écart d'erreur moyen entre le modèle avec l'emploi et la référence, en €/m² ; négatif, l'emploi aide dans ce groupe.

**Résultats propres à H3c :**
- **a. Le coefficient de l'évolution est positif et stable** dans la régression : de 0,003 à 0,008 selon l'année et le type de bien, toujours du même signe, comme attendu (plus d'emplois, prix plus élevé). Celui du niveau est très légèrement négatif (de −0,001 à −0,006). Le **Lasso** met les deux variables d'emploi à 0 dans trois ajustements sur quatre : il ne garde que l'évolution pour les maisons en 2023 (0,0025). Le signal existe, mais il est faible.
- **b. Contre leur propre référence :**
  - le **Lasso** n'apporte rien pour les appartements (gain de 0,0 % : l'emploi est mis à zéro) ; pour les maisons, gain de 1,6 % en 2023 et 7,0 % en 2024 : une année réussie sur deux, **non retenu** ;
  - le **gradient boosting** gagne de 2,7 à 6,7 %, avec des intervalles toujours sous 0 (rang-bisériale de −0,18 à −0,54). Il est **retenu pour les maisons** (3,7 % puis 6,7 %, deux années réussies). Pour les appartements, 2023 reste sous le seuil (2,7 %) : une année sur deux, non retenu ;
  - le **modèle hiérarchique** donne le même résultat que la régression (ICC presque nul). *(À vérifier après relance.)*
- **c. Avec l'emploi de N-2, l'apport disparaît** : gain de 1,9 % et 1,5 % pour les appartements, de −0,8 % et −1,3 % pour les maisons ; aucune année réussie. Pour le prix 2023, l'emploi du 31/12/2021 décrit encore le rebond d'après-Covid (+4,6 % au niveau national) : le signal de ralentissement arrive un an trop tard. Le gain de la version principale repose donc sur une donnée publiée en mai de l'année prévue : c'est la principale limite de H3c.
- **d. Le gain n'est pas réservé aux territoires où l'emploi a beaucoup bougé.**
  - Pour les appartements, l'emploi aide dans tous les quintiles (de −3 à −5 €/m² en 2023, de −9 à −14 €/m² en 2024).
  - Pour les maisons, il aide surtout là où l'emploi a le plus baissé (−4,5 et −9,8 €/m² dans le premier quintile) et de moins en moins quand il augmente.

  Avec l'importance quasi nulle de l'évolution (6.2), cela confirme que l'emploi corrige surtout le **niveau général** des prévisions (le ralentissement de l'emploi annonce celui des prix) plus qu'il ne distingue les territoires.

## 7. Conclusion

- **Contre la naïve** (5.1) : aucun modèle n'est retenu. Tous surestiment les prix pendant le retournement de 2023-2024 et perdent de 4,7 à 67,7 % ; « prix N-1 seul » perd déjà de 11,5 à 66,4 %.
- **Contre « prix N-1 seul »** (5.2 et 6.4 b) : l'emploi apporte un gain réel mais petit. Dans la régression, les intervalles sont toujours sous 0, mais le gain ne dépasse 3 % que pour les appartements en 2024 (de 1,3 à 4,7 % avec le bloc complet). L'évolution porte l'essentiel du gain, le niveau ajoute un peu pour les appartements.
- **Coefficient de l'évolution** (6.4 a) : positif et stable (de 0,003 à 0,008), comme attendu ; le Lasso ne le garde que pour les maisons en 2023.
- **Importance** (6.2) : quasi nulle pour l'évolution. Le gain ne vient pas des écarts entre territoires mais du ralentissement **national** de l'emploi en 2022-2023, qui abaisse toutes les prévisions (confirmé par les quintiles, 6.4 d). L'emploi agit comme une variable « quand » commune à tous, pas comme une variable « où ».
- **Erreurs et contrôle qualité** (6.3) : l'erreur ne se concentre pas dans les territoires aux données estimées ; le secret statistique de l'emploi est sans effet.
- **Calendrier** (6.4 c) : avec l'emploi de N-2, réellement disponible en début d'année, le gain disparaît (au plus 1,9 %, aucune année réussie).
- **Décision pour le modèle complet :**
  - **maisons** : bloc emploi (évolution et niveau) **retenu avec le gradient boosting**, deux années réussies contre son propre « prix N-1 seul » (gains de 3,7 et 6,7 %), non retenu avec la régression, le Lasso et le modèle hiérarchique ;
  - **appartements** : **non retenu** (une année sur deux pour toutes les méthodes : seule 2024 atteint le seuil).
  - Réserve à transmettre : le gain dépend du calendrier de publication (convention « connu en N-1 ») et passe surtout par la tendance nationale.
- **Pour le niveau 2** : ce résultat va dans le sens du diagnostic T1. Ce qui aide, c'est l'information sur la phase du marché (le « quand »), pas la description des territoires. Un indicateur national ou régional de l'emploi, connu plus tôt (séries trimestrielles de l'Urssaf), pourrait porter le même signal sans le délai de publication.
- **Limites** : secteur privé seulement (ni public ni agriculture) ; emplois comptés au lieu de travail ; même valeur d'emploi pour les deux types de bien ; apprentis inclus dans les effectifs (série sans rupture visible, à confirmer dans la méthodologie de l'Urssaf) ; deux années jugées seulement, toutes deux pendant le retournement.